In [1]:
from utils import simulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm


In [2]:
## Simulations Parameter ##############
n = 1000
seed = 42
n_sim = 1000
B_1 = 200


######### Varying Parameters ##########
tau = 12
B_RF = int(n*0.7  * 2)
########################################

X_erwartung = pd.DataFrame({'bmi': [25], 'blood_pressure': [0], 'kreatinkinase': [np.exp(5+1/2)], 'diabetes': [0], 'age': [50]})

params_data_creation = { 'shape_weibull': 1, 'scale_weibull_base': 10000, 'rate_censoring': 0.01,
                        'b_bloodp': -0.405, 'b_diab': -0.4, 'b_age': -0.05, 'b_bmi': -0.01, 'b_kreat': -0.2, 
                        'n': n, 'seed': 0, 'tau': tau}

params_rf = {  'n_estimators':B_RF,                        
                'max_depth':3,
                'min_samples_split':5,
                'max_features': 'log2',
                'random_state':  0,
                'bootstrap':     True,  }


####### ONE Simulation   ########
portion_events_after_cut_train, portion_censored_after_cut_train, portion_no_events_after_cut_train, \
portion_events_after_cut_test, portion_censored_after_cut_test, portion_no_events_after_cut_test, \
    wb_mse_ipcw, wb_cindex_ipcw, wb_y_pred_X_point, \
    rf_mse_ipcw, rf_y_pred_X_point, rf_std_pred_X_point, ijk_std_pred_X_point = simulation(
    seed=seed,tau=tau, data_generation_weibull_parameters=params_data_creation,params_rf=params_rf, X_pred_point=X_erwartung)

print('Train: ')
print(f'Portion of events after cut:     {round(portion_events_after_cut_train*100,2)}%,    n={round(n*0.7*portion_events_after_cut_train,0)}')
print(f'Portion of no events after cut:  {round(portion_no_events_after_cut_train*100,2)}%,    n={round(n*0.7*portion_no_events_after_cut_train,0)}')
print(f'Portion of censored after cut:   {round(portion_censored_after_cut_train*100,2)}%,   n={round(n*0.7*portion_censored_after_cut_train,0)})')
print('Test: ')
print(f'Portion of events after cut:     {round(portion_events_after_cut_test*100,2)}%,    n={round(n*0.7*portion_events_after_cut_test,0)}')
print(f'Portion of no events after cut:  {round(portion_no_events_after_cut_test*100,2)}%,    n={round(n*0.7*portion_no_events_after_cut_test,0)}')
print(f'Portion of censored after cut:   {round(portion_censored_after_cut_test*100,2)}%,   n={round(n*0.7*portion_censored_after_cut_test,0)})')


Train: 
Portion of events after cut:     81.0%,    n=567.0
Portion of no events after cut:  6.57%,    n=46.0
Portion of censored after cut:   12.43%,   n=87.0)
Test: 
Portion of events after cut:     83.67%,    n=586.0
Portion of no events after cut:  7.33%,    n=51.0
Portion of censored after cut:   9.0%,   n=63.0)


In [1]:
from utils import simulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm



### Simulations Parameter ##############
n = 1000
seed = 42
n_sim = 1000

n = 1000
seed = 42
n_sim = 10
B_1 = 20

########## Varying Parameters ##########
tau = 12
B_RF = int(n*0.7  * 2)
#########################################

X_erwartung = pd.DataFrame({'bmi': [25], 'blood_pressure': [0], 'kreatinkinase': [np.exp(5+1/2)], 'diabetes': [0], 'age': [50]})

params_data_creation = { 'shape_weibull': 1, 'scale_weibull_base': 10000, 'rate_censoring': 0.01,
                        'b_bloodp': -0.405, 'b_diab': -0.4, 'b_age': -0.05, 'b_bmi': -0.01, 'b_kreat': -0.2, 
                        'n': n, 'seed': seed, 'tau': tau}

params_rf = {  'n_estimators':B_RF,                        
                'max_depth':3,
                'min_samples_split':5,
                'max_features': 'log2',
                'random_state':  seed,
                'bootstrap':     True,  }


######## Start Simulation   ########

with ProcessPoolExecutor() as executor:
    
    ### Array to store the results
    portion_events_after_cut_train = np.zeros(n_sim)
    portion_censored_after_cut_train = np.zeros(n_sim)
    portion_no_events_after_cut_train = np.zeros(n_sim)
    portion_events_after_cut_test = np.zeros(n_sim)
    portion_censored_after_cut_test = np.zeros(n_sim)
    portion_no_events_after_cut_test = np.zeros(n_sim)
    wb_mse_ipcw = np.zeros(n_sim)
    wb_cindex_ipcw = np.zeros(n_sim)
    wb_y_pred_X_point = np.zeros(n_sim)
    rf_mse_ipcw = np.zeros(n_sim)
    rf_y_pred_X_point = np.zeros(n_sim)
    rf_std_pred_X_point = np.zeros(n_sim)
    ijk_var_pred_X_point = np.zeros(n_sim)
    bootstrap_std_pred_X_point = np.zeros(n_sim)

    futures = [
        executor.submit(
            simulation,
            seed=seed+i,
            tau=tau, 
            data_generation_weibull_parameters=params_data_creation,
            params_rf=params_rf, 
            X_pred_point=X_erwartung,
            B_1 = B_1 
        )
        for i in range(n_sim)
    ]

    for i, future in enumerate(tqdm(futures, desc="Simulations", unit="simulation")):
        _portion_events_after_cut_train, _portion_censored_after_cut_train, _portion_no_events_after_cut_train, \
         _portion_events_after_cut_test, _portion_censored_after_cut_test, _portion_no_events_after_cut_test, \
        _wb_mse_ipcw, _wb_cindex_ipcw, _wb_y_pred_X_point, \
        _rf_mse_ipcw, _rf_y_pred_X_point, _rf_std_pred_X_point, _ijk_var_pred_X_point, _bootstrap_std_pred_X_point  = future.result()

        #Event-Stats Results
        portion_events_after_cut_train[i] = _portion_events_after_cut_train
        portion_censored_after_cut_train[i] = _portion_censored_after_cut_train
        portion_no_events_after_cut_train[i] = _portion_no_events_after_cut_train
        portion_events_after_cut_test[i] = _portion_events_after_cut_test
        portion_censored_after_cut_test[i] = _portion_censored_after_cut_test
        portion_no_events_after_cut_test[i] = _portion_no_events_after_cut_test
        
        #Evaluation Results
        wb_mse_ipcw[i] = _wb_mse_ipcw
        wb_cindex_ipcw[i] = _wb_cindex_ipcw
        rf_mse_ipcw[i] = _rf_mse_ipcw

        #Prediction Results
        wb_y_pred_X_point[i] = _wb_y_pred_X_point[0]
        rf_y_pred_X_point[i] = _rf_y_pred_X_point[0]

        # Standard Deviation Estimates
        rf_std_pred_X_point[i] = _rf_std_pred_X_point
        ijk_var_pred_X_point[i]  = _ijk_var_pred_X_point
        bootstrap_std_pred_X_point[i] = _bootstrap_std_pred_X_point


Simulations: 100%|██████████| 10/10 [00:53<00:00,  5.40s/simulation]


In [12]:
def print_stats_data():

    print('Event-Stats Results:')
    print('Train:')
    print(f'Portion of events after cut:     {round(np.mean(portion_events_after_cut_train)*100,2)}%,   n={round(n*0.7*np.mean(portion_events_after_cut_train),0)}')
    print(f'Portion of no events after cut:  {round(np.mean(portion_no_events_after_cut_train)*100,2)}%,    n={round(n*0.7*np.mean(portion_no_events_after_cut_train),0)}')
    print(f'Portion of censored after cut:   {round(np.mean(portion_censored_after_cut_train)*100,2)}%,   n={round(n*0.7*np.mean(portion_censored_after_cut_train),0)}')
    print('\n')
    print('Test:')
    print(f'Portion of events after cut:     {round(np.mean(portion_events_after_cut_test)*100,2)}%,   n={round(n*0.7*np.mean(portion_events_after_cut_test),0)}')
    print(f'Portion of no events after cut:  {round(np.mean(portion_no_events_after_cut_test)*100,2)}%,    n={round(n*0.7*np.mean(portion_no_events_after_cut_test),0)}')
    print(f'Portion of censored after cut:   {round(np.mean(portion_censored_after_cut_test)*100,2)}%,   n={round(n*0.7*np.mean(portion_censored_after_cut_test),0)}')
    print('\n')

print_stats_data()

Event-Stats Results:
Train:
Portion of events after cut:     81.54%,   n=571.0
Portion of no events after cut:  7.44%,    n=52.0
Portion of censored after cut:   11.01%,   n=77.0


Test:
Portion of events after cut:     81.87%,   n=573.0
Portion of no events after cut:  7.27%,    n=51.0
Portion of censored after cut:   10.87%,   n=76.0




In [13]:
def print_results():
    print('Evaluation Results:')
    print(f'WB C-Index IPCW: {np.mean(wb_cindex_ipcw):.4f}')
    print(f'WB MSE IPCW: {np.mean(wb_mse_ipcw):.4f}')
    print(f'RF MSE IPCW: {np.mean(rf_mse_ipcw):.4f}')
    print('\n')

    print('Prediction Results:')
    print(f'WB Y_pred: {np.mean(wb_y_pred_X_point):.4f}')
    print(f'RF Y_pred: {np.mean(rf_y_pred_X_point):.4f}')
    print('\n')

    print('Standard Deviation:')
    print(f'WB EMP-STD:                 {np.std(wb_y_pred_X_point):.4f}\n')
    print(f'RF EMP-STD:                 {np.std(rf_y_pred_X_point):.4f}')


    non_neg_var_ijk_est = ijk_var_pred_X_point.copy()
    non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
    # erst die wurzel ziehen und dann den mittelwert bilden
    np.mean(np.sqrt(  non_neg_var_ijk_est    ))
    print(f'IJK STD (for RF) Mean-est : {np.mean(np.sqrt(  non_neg_var_ijk_est    )):.4f}')

    non_neg_var_ijk_est = bootstrap_std_pred_X_point.copy()
    non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
    # erst die wurzel ziehen und dann den mittelwert bilden
    np.mean(np.sqrt(  non_neg_var_ijk_est    ))
    print(f'Boot STD (for RF) Mean-est :{np.mean(  bootstrap_std_pred_X_point    ):.4f}')

print_results()

Evaluation Results:
WB C-Index IPCW: 0.6405
WB MSE IPCW: 0.0704
RF MSE IPCW: 0.0694


Prediction Results:
WB Y_pred: 0.9437
RF Y_pred: 0.9484


Standard Deviation:
WB EMP-STD:                 0.0077

RF EMP-STD:                 0.0142
IJK STD (for RF) Mean-est : 0.0109
Boot STD (for RF) Mean-est :0.0104
